In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [10]:
from pathlib import Path

RAW_DIR = Path("/kaggle/working/nixiebomb_raw")

files = list(RAW_DIR.glob("*.txt"))

print("Text files:", len(files))
print("Total size (MB):", round(
    sum(f.stat().st_size for f in files) / (1024 * 1024), 2
))
print("Last file:", files[-1].name if files else "None")

Text files: 980
Total size (MB): 688.59
Last file: codes_0703.txt


In [9]:
from pathlib import Path

RAW_DIR = Path("/kaggle/working/nixiebomb_raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

count = 0
file_index = 0
examples_per_file = 1000

for example in ds:
    comment = example.get("comment", "")
    code = example.get("code", "")

    if not comment.strip() or not code.strip():
        continue

    if count % examples_per_file == 0:
        output_file = RAW_DIR / f"codes_{file_index:04d}.txt"
        file_index += 1

    with output_file.open("a", encoding="utf-8") as f:
        f.write(comment + "\n\n" + code + "\n\n")

    count += 1

    if count % 10000 == 0:
        print(f"Saved {count:,} examples")

print(f"\nFinished: {count:,} examples")
print(f"Files created: {file_index}")
print(f"Location: {RAW_DIR}")

Saved 10,000 examples
Saved 20,000 examples
Saved 30,000 examples
Saved 40,000 examples
Saved 50,000 examples
Saved 60,000 examples
Saved 70,000 examples
Saved 80,000 examples
Saved 90,000 examples
Saved 100,000 examples
Saved 110,000 examples
Saved 120,000 examples
Saved 130,000 examples
Saved 140,000 examples
Saved 150,000 examples
Saved 160,000 examples
Saved 170,000 examples
Saved 180,000 examples
Saved 190,000 examples
Saved 200,000 examples
Saved 210,000 examples
Saved 220,000 examples
Saved 230,000 examples
Saved 240,000 examples
Saved 250,000 examples
Saved 260,000 examples
Saved 270,000 examples
Saved 280,000 examples
Saved 290,000 examples
Saved 300,000 examples
Saved 310,000 examples
Saved 320,000 examples
Saved 330,000 examples
Saved 340,000 examples
Saved 350,000 examples
Saved 360,000 examples
Saved 370,000 examples
Saved 380,000 examples
Saved 390,000 examples
Saved 400,000 examples
Saved 410,000 examples
Saved 420,000 examples
Saved 430,000 examples
Saved 440,000 exampl

KeyboardInterrupt: 

In [8]:
!pip install -q datasets

from datasets import load_dataset

ds = load_dataset(
    "sentence-transformers/codesearchnet",
    "pair",
    split="train",
    streaming=True
)

print("Dataset connected!")
print("First example:")
print(next(iter(ds)))

README.md:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

Dataset connected!
First example:
{'comment': 'Computes the new parent id for the node being moved.\n\n@return int', 'code': "protected function parentId()\n\t{\n\t\tswitch ( $this->position )\n\t\t{\n\t\t\tcase 'root':\n\t\t\t\treturn null;\n\n\t\t\tcase 'child':\n\t\t\t\treturn $this->target->getKey();\n\n\t\t\tdefault:\n\t\t\t\treturn $this->target->getParentId();\n\t\t}\n\t}"}


In [1]:
!git clone --depth 1 --branch Brain https://github.com/Salt-the-Fries/NIXIEBOMB.git

Cloning into 'NIXIEBOMB'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 28 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (28/28), 25.19 KiB | 2.29 MiB/s, done.


In [2]:
%cd /kaggle/working/NIXIEBOMB
!ls

/kaggle/working/NIXIEBOMB
core  data  kaggle  LICENSE  observatory  README.md  requirements.txt


In [4]:
!pip install -q "tokenizers>=0.20" "numpy>=1.26"

In [ ]:
import sys
import subprocess

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "tokenizers>=0.20", "numpy>=1.26"],
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print("Installation output:")
print(result.stdout)
print("Errors:")
print(result.stderr)

In [6]:
import torch
from core.config import ModelConfig
from core.model import NixieBombCore

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

# Create a tiny network with RANDOM initialization
config = ModelConfig.tiny()
model = NixieBombCore(config)

# Give it a small batch of random token IDs
tokens = torch.randint(0, config.vocab_size, (1, 16))

# Run the network
with torch.no_grad():
    result = model(tokens)

print("Output shape:", result["logits"].shape)
print("Parameters:", f"{model.parameter_count():,}")
print("All values finite:", torch.isfinite(result["logits"]).all().item())
print("NIXIEBOMB neural network test complete!")

PyTorch: 2.11.0+cu130
CUDA: True
Output shape: torch.Size([1, 16, 512])
Parameters: 123,712
All values finite: True
NIXIEBOMB neural network test complete!


In [7]:
import os

print("=== KAGGLE DATASETS ===")

for folder, dirs, files in os.walk("/kaggle/input"):
    if files:
        print("\nFolder:", folder)
        print("Files:", len(files))
        print("Examples:", files[:5])

print("\n=== NIXIEBOMB SOURCE FILES ===")

for folder, dirs, files in os.walk("/kaggle/working/NIXIEBOMB/core"):
    for file in files:
        if file.endswith((".py", ".md")):
            print(os.path.join(folder, file))

=== KAGGLE DATASETS ===

=== NIXIEBOMB SOURCE FILES ===
/kaggle/working/NIXIEBOMB/core/prepare_data.py
/kaggle/working/NIXIEBOMB/core/inspect.py
/kaggle/working/NIXIEBOMB/core/model.py
/kaggle/working/NIXIEBOMB/core/config.py
/kaggle/working/NIXIEBOMB/core/observatory.py
/kaggle/working/NIXIEBOMB/core/__init__.py
/kaggle/working/NIXIEBOMB/core/tokenizer.py
/kaggle/working/NIXIEBOMB/core/train.py
/kaggle/working/NIXIEBOMB/core/tests/test_observatory.py
/kaggle/working/NIXIEBOMB/core/tests/test_model.py
/kaggle/working/NIXIEBOMB/core/tests/test_tokenizer.py


In [11]:
from pathlib import Path

RAW_DIR = Path("/kaggle/working/nixiebomb_raw")
files = sorted(RAW_DIR.glob("*.txt"))

print("Files:", len(files))
print("First 5:", [f.name for f in files[:5]])
print("Last 5:", [f.name for f in files[-5:]])

for f in files[-3:]:
    print(f"\n{f.name}: {f.stat().st_size / 1024:.1f} KB")
    print(f.read_text(encoding="utf-8")[:250])

Files: 980
First 5: ['codes_0000.txt', 'codes_0001.txt', 'codes_0002.txt', 'codes_0003.txt', 'codes_0004.txt']
Last 5: ['codes_0975.txt', 'codes_0976.txt', 'codes_0977.txt', 'codes_0978.txt', 'codes_0979.txt']

codes_0977.txt: 705.3 KB
Dispatches an action inmediatelly.

@return {Promise} A promise to be resolved when all the callbacks have finised.

function(){
		var me = this,
			dispatchArguments = arguments,
			promises = []
		;

		this._promises = [];

		// A closure is needed

codes_0978.txt: 728.6 KB
<!-- begin-user-doc -->
<!-- end-user-doc -->
@generated

@Override
	public EClass getIfcDoorLiningProperties() {
		if (ifcDoorLiningPropertiesEClass == null) {
			ifcDoorLiningPropertiesEClass = (EClass) EPackage.Registry.INSTANCE.getEPackage(Ifc4Pa

codes_0979.txt: 87.4 KB
Matrix triangular solve op.

def MatrixTriangularSolve(a, rhs, lower, adj):
    
    trans = 0 if not adj else 2

    r = np.empty(rhs.shape).astype(a.dtype)
    for coord in np.ndindex(a.shape[:-2]):
        po

In [12]:
!python -m core.prepare_data --help

usage: prepare_data.py [-h] --input INPUT [--out OUT]
                       [--vocab-size VOCAB_SIZE] [--seed SEED]
                       [--val-fraction VAL_FRACTION]

Prepare a tokenizer and token-ID corpus from scratch

options:
  -h, --help            show this help message and exit
  --input INPUT
  --out OUT
  --vocab-size VOCAB_SIZE
  --seed SEED
  --val-fraction VAL_FRACTION


In [13]:
%cd /kaggle/working/NIXIEBOMB

!python -m core.prepare_data \
    --input /kaggle/working/nixiebomb_raw \
    --out /kaggle/working/nixiebomb_processed \
    --vocab-size 16384 \
    --val-fraction 0.05

/kaggle/working/NIXIEBOMB
[00:00:05] Tokenize words                 ██████████████████ 2444818  /  2444818[00:00:00] Tokenize words                 ██████████████████ 0        /        0
[00:00:06] Count pairs                    ██████████████████ 2444818  /  2444818
[00:00:32] Compute merges                 ██████████████████ 16124    /    16124
{
  "files_total": 980,
  "train_files": 931,
  "val_files": 49,
  "train_tokens": 199205652,
  "val_tokens": 10380674,
  "tokenizer": {
    "kind": "byte-level-bpe",
    "trained_from_scratch": true,
    "vocab_size": 16384,
    "special_tokens": [
      "<|unk|>",
      "<|bos|>",
      "<|eos|>",
      "<|pad|>"
    ],
    "tokenizer_file": "tokenizer.json"
  },
  "seed": 1337,
  "secret_patterns_redacted": true,
  "note": "Review source licenses and ownership before training. Secret scanning is heuristic, not a guarantee."
}


In [15]:
from pathlib import Path

paths = {
    "Original dataset": Path("/kaggle/working/nixiebomb_raw"),
    "Prepared dataset": Path("/kaggle/working/nixiebomb_processed"),
}

for name, path in paths.items():
    print(f"\n{name}:")
    print("Exists:", path.exists())

    if path.exists():
        for filename in ["train.bin", "val.bin", "tokenizer.json", "manifest.json"]:
            f = path / filename
            if f.exists():
                print(f"  {filename}: {f.stat().st_size / (1024**2):.2f} MB")


Original dataset:
Exists: True

Prepared dataset:
Exists: True
  train.bin: 379.95 MB
  val.bin: 19.80 MB
  tokenizer.json: 1.06 MB
  manifest.json: 0.00 MB
